In [7]:
import requests
import os
import time

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project/tempdata"
BASE_URL   = "https://www.ncei.noaa.gov/pub/data/daily-grids/v1-0-0/averages"

YEAR_MIN = 2012
YEAR_MAX = 2024

# For the presentation, prioritize summer months first
# then fill in the rest — comment out MONTHS_PRIORITY if you want all at once
MONTHS_ALL    = list(range(1, 13))
MONTHS_SUMMER = [6, 7, 8]

# Change to MONTHS_ALL if you want everything in one run
# For April 14 presentation, MONTHS_ALL is fine — 156 files at ~1MB each
MONTHS = MONTHS_ALL

# ── DOWNLOAD ─────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

total    = len(range(YEAR_MIN, YEAR_MAX + 1)) * len(MONTHS)
done     = 0
skipped  = 0
failed   = []

for year in range(YEAR_MIN, YEAR_MAX + 1):
    for month in MONTHS:
        mm       = str(month).zfill(2)
        filename = f"prcp-{year}{mm}-cty-scaled.csv"
        url      = f"{BASE_URL}/{year}/{filename}"
        outpath  = os.path.join(OUTPUT_DIR, filename)

        # Skip if already downloaded
        if os.path.exists(outpath) and os.path.getsize(outpath) > 10000:
            skipped += 1
            done += 1
            continue

        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(outpath, "wb") as f:
                    f.write(r.content)
                done += 1
                print(f"  ✓ {filename} ({len(r.content)/1024:.0f} KB)  [{done}/{total}]")
            else:
                failed.append((url, r.status_code))
                print(f"  ✗ {filename} — HTTP {r.status_code}")
                done += 1
        except Exception as e:
            failed.append((url, str(e)))
            print(f"  ✗ {filename} — ERROR: {e}")
            done += 1

        # Be polite to the server
        time.sleep(0.3)

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"Done. {done} files processed, {skipped} already existed.")
if failed:
    print(f"\nFailed downloads ({len(failed)}):")
    for url, reason in failed:
        print(f"  {url} — {reason}")
else:
    print("No failures.")

  ✓ prcp-201201-cty-scaled.csv (1068 KB)  [1/156]
  ✓ prcp-201202-cty-scaled.csv (1068 KB)  [2/156]
  ✓ prcp-201203-cty-scaled.csv (1068 KB)  [3/156]
  ✓ prcp-201204-cty-scaled.csv (1068 KB)  [4/156]
  ✓ prcp-201205-cty-scaled.csv (1068 KB)  [5/156]
  ✓ prcp-201206-cty-scaled.csv (1068 KB)  [6/156]
  ✓ prcp-201207-cty-scaled.csv (1068 KB)  [7/156]
  ✓ prcp-201208-cty-scaled.csv (1068 KB)  [8/156]
  ✓ prcp-201209-cty-scaled.csv (1068 KB)  [9/156]
  ✓ prcp-201210-cty-scaled.csv (1068 KB)  [10/156]
  ✓ prcp-201211-cty-scaled.csv (1068 KB)  [11/156]
  ✓ prcp-201212-cty-scaled.csv (1068 KB)  [12/156]
  ✓ prcp-201301-cty-scaled.csv (1068 KB)  [13/156]
  ✓ prcp-201302-cty-scaled.csv (1068 KB)  [14/156]
  ✓ prcp-201303-cty-scaled.csv (1068 KB)  [15/156]
  ✓ prcp-201304-cty-scaled.csv (1068 KB)  [16/156]
  ✓ prcp-201305-cty-scaled.csv (1068 KB)  [17/156]
  ✓ prcp-201306-cty-scaled.csv (1068 KB)  [18/156]
  ✓ prcp-201307-cty-scaled.csv (1068 KB)  [19/156]
  ✓ prcp-201308-cty-scaled.csv (1068 KB)

In [46]:
import pandas as pd
import numpy as np
import glob
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
NOAA_DIR    = "/Users/aliabonanno/public_policy_project/tempdata"
OUTPUT_PATH = "/Users/aliabonanno/public_policy_project/noaa_heat_panel.csv"

YEAR_MIN   = 2012
YEAR_MAX   = 2024
THRESHOLDS = [90, 95, 100]

def c_to_f(c):
    return c * 9/5 + 32

# ── LOAD NCEI -> CENSUS FIPS CROSSWALK ───────────────────────────────────────
xwalk_path = os.path.join(NOAA_DIR, "us-state-codes_ncei-to-fips.csv")
xwalk = pd.read_csv(xwalk_path, dtype=str)
xwalk["NCEI_code"] = xwalk["NCEI_code"].str.zfill(2)
xwalk["FIPS_code"] = xwalk["FIPS_code"].str.zfill(2)
print(f"Loaded crosswalk: {len(xwalk)} states")

# ── LOOP ─────────────────────────────────────────────────────────────────────
all_months = []
files = sorted(glob.glob(os.path.join(NOAA_DIR, "tmax-*-cty-scaled.csv")))
print(f"Found {len(files)} files.")

for fpath in files:
    basename = os.path.basename(fpath)
    yyyymm = basename.split("-")[1]
    year  = int(yyyymm[:4])
    month = int(yyyymm[4:])

    if year < YEAR_MIN or year > YEAR_MAX:
        continue

    day_cols  = [f"day{i}" for i in range(1, 32)]
    col_names = ["type", "FIPS", "county_name", "year_col",
                 "month_col", "variable"] + day_cols

    df = pd.read_csv(fpath, header=None, names=col_names, dtype={1: str})
    df = df[df["variable"] == "TMAX"].copy()
    df["FIPS"] = df["FIPS"].str.strip().str.zfill(5)

    # ── REMAP NCEI STATE CODE TO CENSUS FIPS ─────────────────────────────
    df["ncei_state"]  = df["FIPS"].str[:2]
    df["county_code"] = df["FIPS"].str[2:]
    df = df.merge(xwalk[["NCEI_code", "FIPS_code"]],
                  left_on="ncei_state", right_on="NCEI_code", how="left")
    df["FIPS"] = df["FIPS_code"] + df["county_code"]
    df = df.dropna(subset=["FIPS"])

    # Convert to Fahrenheit
    for col in day_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = c_to_f(df[col])

    # Count days above each threshold
    daily_f = df[day_cols].values.astype(float)
    for thresh in THRESHOLDS:
        df[f"days_above_{thresh}f"] = np.nansum(
            daily_f > thresh, axis=1).astype(int)

    df["year"]  = year
    df["month"] = month

    keep = ["FIPS", "year", "month"] + [f"days_above_{t}f" for t in THRESHOLDS]
    all_months.append(df[keep])

# ── STACK AND SAVE ────────────────────────────────────────────────────────────
panel = pd.concat(all_months, ignore_index=True)
panel = panel.sort_values(["FIPS", "year", "month"]).reset_index(drop=True)

print(f"\nFinal panel: {len(panel)} rows")
print(f"Years covered: {sorted(panel['year'].unique())}")
print(f"Unique counties: {panel['FIPS'].nunique()}")

# Sanity check: Autauga July 2020
autauga = panel[(panel["FIPS"] == "01001") &
                (panel["year"] == 2020) &
                (panel["month"] == 7)]
print(f"\nAutauga County (01001) July 2020:")
print(autauga.to_string(index=False))

# Sanity check: Iowa county — should now be 19xxx
iowa = panel[panel["FIPS"].str.startswith("19")]
print(f"\nIowa counties (should be 19xxx): {iowa['FIPS'].nunique()}")
print(iowa["FIPS"].unique()[:5])

# National summer averages
summer = panel[panel["month"].isin([6, 7, 8])]
print(f"\nNational mean days above 90F in summer: {summer['days_above_90f'].mean():.2f}")
print(f"National mean days above 95F in summer: {summer['days_above_95f'].mean():.2f}")

panel.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Loaded crosswalk: 48 states
Found 169 files.

Final panel: 484692 rows
Years covered: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Unique counties: 3107

Autauga County (01001) July 2020:
 FIPS  year  month  days_above_90f  days_above_95f  days_above_100f
01001  2020      7              22               4                0

Iowa counties (should be 19xxx): 99
['19001' '19003' '19005' '19007' '19009']

National mean days above 90F in summer: 9.84
National mean days above 95F in summer: 3.33

Saved to /Users/aliabonanno/public_policy_project/noaa_heat_panel.csv


In [17]:
import pandas as pd

fpath = "/Users/aliabonanno/public_policy_project/tempdata/tmax-202001-cty-scaled.csv"

# Read with NO name assignment, just see raw structure
raw = pd.read_csv(fpath, header=None, nrows=3)
print(raw.shape)
print(raw.iloc[0].tolist())  # what is actually in row 0?
print(raw.iloc[1].tolist())  # what is in row 1?

(3, 37)
['cty', 1001, 'AL: Autauga', 2020, 1, 'TMAX', 14.16, 14.45, 21.53, 19.3, 15.31, 14.79, 18.87, 16.41, 18.82, 18.91, 23.29, 23.28, 20.6, 19.4, 20.67, 20.87, 20.88, 19.75, 15.85, 11.06, 8.98, 7.38, 10.53, 11.34, 15.47, 12.77, 12.25, 14.85, 17.47, 12.16, 16.26]
['cty', 1003, 'AL: Baldwin County', 2020, 1, 'TMAX', 16.85, 18.01, 23.3, 21.47, 17.38, 16.45, 20.46, 18.32, 18.48, 20.32, 22.49, 24.27, 19.97, 23.25, 25.68, 25.64, 23.71, 21.1, 19.98, 15.23, 9.11, 6.79, 10.92, 14.12, 18.28, 16.31, 11.79, 14.55, 18.71, 12.78, 17.18]


In [31]:
import pandas as pd

fpath = "/Users/aliabonanno/public_policy_project/tempdata/tmax-202001-cty-scaled.csv"
df = pd.read_csv(fpath, header=None, nrows=20)
print(df.shape)
print(df[[0,1,3,4]].to_string())  # type, FIPS, year, month

(20, 37)
      0     1     3  4
0   cty  1001  2020  1
1   cty  1003  2020  1
2   cty  1005  2020  1
3   cty  1007  2020  1
4   cty  1009  2020  1
5   cty  1011  2020  1
6   cty  1013  2020  1
7   cty  1015  2020  1
8   cty  1017  2020  1
9   cty  1019  2020  1
10  cty  1021  2020  1
11  cty  1023  2020  1
12  cty  1025  2020  1
13  cty  1027  2020  1
14  cty  1029  2020  1
15  cty  1031  2020  1
16  cty  1033  2020  1
17  cty  1035  2020  1
18  cty  1037  2020  1
19  cty  1039  2020  1


In [16]:
import pandas as pd
import numpy as np
import glob
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
NOAA_DIR     = "/Users/aliabonanno/public_policy_project/tempdata"
OUTPUT_PATH  = "/Users/aliabonanno/public_policy_project/noaa_panel_final.csv"

YEAR_MIN = 2015
YEAR_MAX = 2024

# ── LOAD NCEI -> CENSUS FIPS CROSSWALK ───────────────────────────────────────
xwalk_path = os.path.join(NOAA_DIR, "us-state-codes_ncei-to-fips.csv")
xwalk = pd.read_csv(xwalk_path, dtype=str)
xwalk["NCEI_code"] = xwalk["NCEI_code"].str.zfill(2)
xwalk["FIPS_code"] = xwalk["FIPS_code"].str.zfill(2)
print(f"Loaded crosswalk: {len(xwalk)} states")

# ── DOREMUS TEMPERATURE BINS (Celsius) ───────────────────────────────────────
BIN_EDGES  = [-np.inf, -5, 0, 5, 10, 15, 20, 25, 30, np.inf]
BIN_LABELS = ["bin1_under_neg5", "bin2_neg5to0", "bin3_0to5",
              "bin4_5to10",      "bin5_10to15",  "bin6_15to20",
              "bin7_20to25",     "bin8_25to30",  "bin9_over30"]

# ── HELPER: READ AND REMAP FIPS ───────────────────────────────────────────────
def read_noaa_file(fpath, variable_name):
    day_cols  = [f"day{i}" for i in range(1, 32)]
    col_names = ["type", "FIPS", "county_name", "year_col",
                 "month_col", "variable"] + day_cols

    df = pd.read_csv(fpath, header=None, names=col_names, dtype={1: str})
    df = df[df["variable"] == variable_name].copy()

    if len(df) == 0:
        return None

    # Pad and remap FIPS
    df["FIPS"] = df["FIPS"].str.strip().str.zfill(5)
    df["ncei_state"]  = df["FIPS"].str[:2]
    df["county_code"] = df["FIPS"].str[2:]
    df = df.merge(xwalk[["NCEI_code", "FIPS_code"]],
                  left_on="ncei_state", right_on="NCEI_code", how="left")
    df["FIPS"] = df["FIPS_code"] + df["county_code"]
    df = df.dropna(subset=["FIPS"])

    # Convert day columns to numeric
    for col in day_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df, day_cols

# ── LOOP OVER MONTHS ──────────────────────────────────────────────────────────
all_months = []

for year in range(YEAR_MIN, YEAR_MAX + 1):
    for month in range(1, 13):
        mm = str(month).zfill(2)

        # ── TAVG ──────────────────────────────────────────────────────────
        tavg_path = os.path.join(NOAA_DIR, f"tavg-{year}{mm}-cty-scaled.csv")
        prcp_path = os.path.join(NOAA_DIR, f"prcp-{year}{mm}-cty-scaled.csv")

        if not os.path.exists(tavg_path):
            print(f"  WARNING: Missing tavg {year}-{mm}")
            continue
        if not os.path.exists(prcp_path):
            print(f"  WARNING: Missing prcp {year}-{mm}")
            continue

        result = read_noaa_file(tavg_path, "TAVG")
        if result is None:
            print(f"  WARNING: No TAVG data in {year}-{mm}")
            continue
        df_tavg, day_cols = result

        # Count days in each temperature bin
        daily_temp = df_tavg[day_cols].values.astype(float)
        for i, label in enumerate(BIN_LABELS):
            low  = BIN_EDGES[i]
            high = BIN_EDGES[i + 1]
            df_tavg[label] = np.nansum(
                (daily_temp > low) & (daily_temp <= high), axis=1
            ).astype(int)

        df_tavg["year"]  = year
        df_tavg["month"] = month

        temp_keep = ["FIPS", "year", "month"] + BIN_LABELS
        df_tavg = df_tavg[temp_keep]

        # ── PRCP ──────────────────────────────────────────────────────────
        result_p = read_noaa_file(prcp_path, "PRCP")
        if result_p is None:
            print(f"  WARNING: No PRCP data in {year}-{mm}")
            df_merged = df_tavg.copy()
            df_merged["prcp_monthly_mm"] = np.nan
        else:
            df_prcp, day_cols_p = result_p
            daily_prcp = df_prcp[day_cols_p].values.astype(float)

            # Monthly total precipitation (sum of daily values)
            # Replace negative values (sentinel missing values) with NaN before summing
            daily_prcp_clean = np.where(daily_prcp < 0, np.nan, daily_prcp)
            df_prcp["prcp_monthly_mm"] = np.nansum(daily_prcp_clean, axis=1)
            df_prcp["year"]  = year
            df_prcp["month"] = month
            df_prcp = df_prcp[["FIPS", "year", "month", "prcp_monthly_mm"]]

            df_merged = df_tavg.merge(df_prcp, on=["FIPS","year","month"],
                                      how="left")

        all_months.append(df_merged)

    print(f"  {year}: processed")

# ── STACK AND SAVE ────────────────────────────────────────────────────────────
panel = pd.concat(all_months, ignore_index=True)
panel = panel.sort_values(["FIPS", "year", "month"]).reset_index(drop=True)

print(f"\nFinal panel: {len(panel)} rows")
print(f"Years: {sorted(panel['year'].unique())}")
print(f"Unique counties: {panel['FIPS'].nunique()}")
print(f"Missing prcp: {panel['prcp_monthly_mm'].isna().sum()}")

# Sanity check: Autauga July 2020
autauga = panel[(panel["FIPS"] == "01001") &
                (panel["year"] == 2020) &
                (panel["month"] == 7)]
print(f"\nAutauga County July 2020:")
print(autauga[["FIPS","year","month",
               "bin7_20to25","bin8_25to30","bin9_over30",
               "prcp_monthly_mm"]].to_string(index=False))

# Summer bin distribution
summer = panel[panel["month"].isin([6, 7, 8])]
print(f"\nNational summer mean days per bin:")
for label in BIN_LABELS[-3:]:
    print(f"  {label}: {summer[label].mean():.2f}")

panel.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Loaded crosswalk: 48 states
  2015: processed
  2016: processed
  2017: processed
  2018: processed
  2019: processed
  2020: processed
  2021: processed
  2022: processed
  2023: processed
  2024: processed

Final panel: 372840 rows
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Unique counties: 3107
Missing prcp: 0

Autauga County July 2020:
 FIPS  year  month  bin7_20to25  bin8_25to30  bin9_over30  prcp_monthly_mm
01001  2020      7            0           31            0           136.17

National summer mean days per bin:
  bin7_20to25: 11.71
  bin8_25to30: 11.79
  bin9_over30: 1.21

Saved to /Users/aliabonanno/public_policy_project/noaa_panel_final.csv


In [11]:
import pandas as pd

fpath = "/Users/aliabonanno/public_policy_project/tempdata/tavg-202007-cty-scaled.csv"
day_cols = [f"day{i}" for i in range(1, 32)]
col_names = ["type", "FIPS", "county_name", "year_col", "month_col", "variable"] + day_cols

df = pd.read_csv(fpath, header=None, names=col_names, dtype={1: str})
df["FIPS"] = df["FIPS"].str.strip().str.zfill(5)

# Check Autauga (NCEI code for Alabama is 01)
autauga = df[df["FIPS"] == "01001"]
print(f"Variable values: {autauga['variable'].unique()}")
print(f"\nDay values:")
print(autauga[day_cols].values[0])

Variable values: ['TAVG']

Day values:
[27.05 25.99 26.67 27.96 27.65 27.66 27.2  26.84 26.68 27.41 28.37 27.41
 27.25 27.68 28.7  28.27 27.56 28.89 29.18 29.44 29.1  28.52 28.07 27.74
 27.08 27.62 27.33 27.39 27.69 26.92 27.84]
